In [25]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test_Iceberg_Query").getOrCreate()

print("=== DANH SÁCH CÁC BẢNG TRONG TIKI ===")
spark.sql("SHOW TABLES IN local_catalog.tiki_bronze").show(truncate=False)


=== DANH SÁCH CÁC BẢNG TRONG TIKI ===
+-----------+------------+-----------+
|namespace  |tableName   |isTemporary|
+-----------+------------+-----------+
|tiki_bronze|products_raw|false      |
+-----------+------------+-----------+



In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test_Iceberg_Query").getOrCreate()

print("=== 10 SẢN PHẨM MỚI NHẤT TRONG ICEBERG ===")
query_1 = """
    SELECT id, name, price, brand_name, crawl_date
    FROM local_catalog.tiki_bronze.products_raw
    LIMIT 10
"""
spark.sql(query_1).show()

print("=== THỐNG KÊ SỐ LƯỢNG THEO DANH MỤC ===")
query_2 = """
    SELECT category_id, COUNT(id) as total_products 
    FROM local_catalog.tiki_bronze.products_raw
    GROUP BY category_id
    order by total_products desc
"""
spark.sql(query_2).show()


=== 10 SẢN PHẨM MỚI NHẤT TRONG ICEBERG ===
+---------+--------------------+------+--------------------+----------+
|       id|                name| price|          brand_name|crawl_date|
+---------+--------------------+------+--------------------+----------+
|279093481|Mặt Nạ Vàng 24k C...|460000|              Karmel|2026-05-22|
|278842139|Mặt nạ matcha làm...|690000|              SU:M37|2026-05-22|
|278786294|Mặt Nạ Đất Sét Qu...|349000|             Aperire|2026-05-22|
|276130106|Mặt nạ bí đao Coc...|145000|The Cocoon Origin...|2026-05-22|
|275969670|Mặt nạ nghệ Hưng ...|144000|The Cocoon Origin...|2026-05-22|
|275969613|Mặt nạ nghệ Hưng ...|258000|The Cocoon Origin...|2026-05-22|
|275969554|Mặt nạ bí đao Coc...|288000|The Cocoon Origin...|2026-05-22|
|275718046|Mặt nạ nghệ Hưng ...|345000|The Cocoon Origin...|2026-05-22|
|275431992|Mặt Nạ Bột Dẻo Ad...| 59000|                Adel|2026-05-22|
|275358414|Mặt Nạ Bột Dẻo TB...| 49000|                 TBM|2026-05-22|
+---------+----------

In [20]:
spark.sql("select * from local_catalog.tiki_bronze.products_raw").show()

+---------+-------------+--------------------+--------------------+------+--------------+--------+-------------+--------------------+--------------+------------+--------------------+-----------+-------------+----------+--------------------+--------------------+
|       id|          sku|                name|             url_key| price|original_price|discount|discount_rate|          brand_name|rating_average|review_count|       thumbnail_url|category_id|quantity_sold|crawl_date|           loaded_at|         source_file|
+---------+-------------+--------------------+--------------------+------+--------------+--------+-------------+--------------------+--------------+------------+--------------------+-----------+-------------+----------+--------------------+--------------------+
|279093481|1822096185602|Mặt Nạ Vàng 24k C...|mat-na-vang-24k-c...|460000|        460000|       0|            0|              Karmel|           0.0|           0|https://salt.tiki...|      11709|            0|2026-0

In [23]:
spark.sql("select * from local_catalog.tiki_bronze.products_raw where id = 248681254 order by crawl_date").show(truncate=False)

+---------+-------------+------------------------------+---------------------------------------+------+--------------+--------+-------------+------------+--------------+------------+-----------------------------------------------------------------------------------------------+-----------+-------------+----------+--------------------------+--------------------------------------+
|id       |sku          |name                          |url_key                                |price |original_price|discount|discount_rate|brand_name  |rating_average|review_count|thumbnail_url                                                                                  |category_id|quantity_sold|crawl_date|loaded_at                 |source_file                           |
+---------+-------------+------------------------------+---------------------------------------+------+--------------+--------+-------------+------------+--------------+------------+------------------------------------------------------

In [ ]:
query = """
        select brand_name, round(avg(price), 2)
        from local_catalog.tiki.products
        group by brand_name
"""
spark.sql(query).show()

In [ ]:
query = """
        select count(distinct brand_name) as count_brand
        from local_catalog.tiki.products
"""
spark.sql(query).show()

In [ ]:
query = """
        select count(*) as count_product_of_durex
        from local_catalog.tiki.products
        where brand_name = 'Durex'
"""
spark.sql(query).show()

In [4]:
print("=== DANH SÁCH DATABASE ===")
spark.sql("SHOW DATABASES IN local_catalog").show(truncate=False)

=== DANH SÁCH DATABASE ===
+-----------+
|namespace  |
+-----------+
|default    |
|tiki_bronze|
+-----------+



In [ ]:
print("=== DANH SÁCH CÁC BẢNG TRONG TIKI ===")
spark.sql("SHOW TABLES IN local_catalog.tiki").show(truncate=False)

In [ ]:
print("=== CẤU TRÚC BẢNG PRODUCTS ===")
spark.sql("DESCRIBE local_catalog.tiki.products").show(truncate=False)

In [ ]:
print("=== LỊCH SỬ CÁC LẦN GHI DỮ LIỆU ===")
spark.sql("SELECT * FROM local_catalog.tiki.products.history").show(truncate=False)

print("=== CHI TIẾT SNAPSHOTS ===")
spark.sql("SELECT committed_at, snapshot_id, operation FROM local_catalog.tiki.products.snapshots").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT count(id) as so_sp_bien_dong, crawl_date 
    FROM local_catalog.tiki.price_history 
    WHERE crawl_date = '2026-05-15'
    GROUP BY crawl_date
""").show()

In [ ]:
spark.sql("""
    select a.id, b.name, b.price as new_price, a.price as old_price
    from local_catalog.tiki.price_history a
    join local_catalog.tiki.products b
    on a.id = b.id
    where a.price != b.price
""").show(100)

In [ ]:
spark.sql("""
    select * from local_catalog.tiki.price_history 
    where crawl_date = '2026-05-14'
""").show()

In [ ]:
spark.sql("""
    SELECT count(*) as so_luong_sp_moi 
    FROM local_catalog.tiki.price_history 
    WHERE crawl_date = '2026-05-15'
""").show()

In [ ]:
spark.sql("""
    SELECT count(*) as so_luong_sp_moi 
    FROM local_catalog.tiki.price_history 
""").show()

In [ ]:
spark.sql("""
    SELECT count(*) as so_luong_sp
    FROM local_catalog.tiki.products
""").show()

In [ ]:
spark.sql("""
    SELECT 
        a.id, 
        b.name, 
        a.price as gia_ghi_nhan_cdc,
        b.price as gia_goc
    FROM local_catalog.tiki.price_history a
    JOIN local_catalog.tiki.products b ON a.id = b.id
    WHERE a.crawl_date = '2026-05-15'
""").show(50, truncate=False)